In [ ]:
%reload_ext autoreload
%autoreload 2
%cd ~/erc-src/cuneiform-ocr-sign-alignment-worktree # Change to the project directory
%env PATH=$HOME/.local/bin:$PATH

import os
import cv2
import numpy as np
import torch
from dotenv import load_dotenv

from sign_alignment.detector import ModelConfig, TabletImageDetector
from sign_alignment.data_source import LocalDataSource, LocalTestDataSource, SubtabletEBLAPISource
from sign_alignment.visualizer import ColorConfig

ANNOTATIONS_DIR = os.path.expanduser("~/erc-work-data/data-of-cuneiform-ocr-data/filtered_annotations")
CONFIG_FILE = "configs/detr.py"
CHECKPOINT_FILE = os.path.expanduser("~/erc-work-data/retrained_models/detr-173/epoch_1000.pth")
TEST_DATA_DIR = os.path.expanduser("~/erc-work-data/ready-for-training/coco-recognition-2025-09/data/coco")
# # temporal change
# CHECKPOINT_FILE = os.path.expanduser("~/epoch_1000.pth")
# ANNOTATIONS_DIR = os.path.expanduser("~/filtered_annotations")
# # ---
SCORE_THRESHOLD = 0.5
OUTPUT_DIR = "alignment_results"
SAMPLE_LIMIT = 5

load_dotenv() # MONGODB_URI should be loaded from .env
MONGODB_URI = os.getenv('MONGODB_URI', 'YOUR_MONGODB_URI')
CANONICAL_FEATURE_DIR = "~/erc-work-data/signs_alignment_data/precompute_feautures/"
if not MONGODB_URI or MONGODB_URI == 'YOUR_MONGODB_URI':
    raise ValueError("MONGODB_URI is required for Mongo-backed canonical sign images")


In [ ]:
from sign_alignment.pipeline import (
    CropContext, PipelineConfig, DEBUG_STEPS_WITH_DIFT,
    Runner, VisOptions,
)
from sign_alignment.dift_model import DiftConfig
from sign_alignment.dift_align import DiftAlignmentConfig, EBLMongoCanonicalSource

_DIFT_REPO = os.path.expanduser("~/erc-src/ProtoSnap")
dift_cfg = DiftConfig(repo_root=_DIFT_REPO)


def make_ebl_mongo_canonical_source(period, config):
    return EBLMongoCanonicalSource(
        mongodb_uri=MONGODB_URI,
        period=period,
        db_name="ebl",
        form="canonical1",
        require_centroid=True,
    )

model_config = ModelConfig(
    config_file=CONFIG_FILE,
    checkpoint_file=CHECKPOINT_FILE,
    device='auto'
)

if 'tablet_detector' not in globals() or getattr(tablet_detector, 'model', None) is None:
    tablet_detector = TabletImageDetector(
        model_config=model_config,
        score_threshold=SCORE_THRESHOLD,
        keep_crops=True,
        is_crop_itself=False, # set to false if testing signs alignment.
    )
else:
    print("Reusing existing tablet_detector instance.")

crop_context = CropContext( # remember to set is_crop_itself to False
    config=PipelineConfig(
        model_config=model_config,
        tablet_detector=tablet_detector,
        local_source=LocalDataSource(ANNOTATIONS_DIR),
        color_config=ColorConfig,
        output_dir=OUTPUT_DIR,
        img_idx=1,
        dift=dift_cfg,
        dift_alignment=DiftAlignmentConfig(affine_probe_padding_ratio=0.1),
        canonical_source_factory=make_ebl_mongo_canonical_source,
        canonical_feature_dir=CANONICAL_FEATURE_DIR,
    )
)

crop_context_using_test_data = CropContext( # remember to set is_crop_itself to True
    config=PipelineConfig(
        model_config=model_config,
        tablet_detector=tablet_detector,
        local_source=LocalTestDataSource(TEST_DATA_DIR),
        api_source=SubtabletEBLAPISource(),
        color_config=ColorConfig,
        output_dir=OUTPUT_DIR,
        img_idx=1,
        dift=dift_cfg,
        dift_alignment=DiftAlignmentConfig(affine_probe_padding_ratio=0.1),
        canonical_source_factory=make_ebl_mongo_canonical_source,
        canonical_feature_dir=CANONICAL_FEATURE_DIR,
    )
)

vis = VisOptions(info=True, display=True, save=True)

runner = Runner(
    context=crop_context, # switch context here
    steps=DEBUG_STEPS_WITH_DIFT,
    vis=vis,
)


In [ ]:
import sign_alignment.pipeline as pp

runner.choose_sample(9)  # index 0 = NBC.4020, index 9 = HS.2086
# Load image, ground truth, and sign text from API in one step
runner.choose_sample(name="YBC.12860")  # switch sample here
runner.run_single_step(pp.step_load_data)


In [ ]:
# detect signs (full image + chosen exp_image crop)
runner.choose_crop(5)  # switch crop index here (0 = full image, 1+ = exp_image crops)
runner.run_single_step(pp.step_detect_signs)

In [ ]:
# transform GT boxes into sub-image coordinates and visualize
runner.run_single_step(pp.step_transform_gt_to_img)


In [ ]:
# compute average detection box dimensions
runner.run_single_step(pp.step_compute_statistics)

In [ ]:
# create detection and text sub-tablets
runner.run_single_step(pp.step_create_box_sets)

In [ ]:
# DBSCAN row detection on detection sub-tablet (also reports text sub-tablet rows)
runner.run_single_step(pp.step_detect_rows)

In [ ]:
# DP row matching between detection and text sub-tablets
runner.run_single_step(pp.step_match_rows)

In [ ]:
# visualize detection rows with D# / D#→R# labels
runner.run_single_step(pp.step_visualize_detection_rows)

In [ ]:
# within-row sign matching for each matched row pair
runner.run_single_step(pp.step_match_signs_in_rows)

In [ ]:
# align text rows onto detection rows using regression baselines
# result stored in aligned_boxes
runner.run_single_step(pp.step_align_text_rows)


In [ ]:
# build sign match info; draw text mapping, side-by-side composite, alignment diagnostic
runner.run_single_step(pp.step_build_sign_match_info)

In [ ]:
# position offset analysis: coarse-aligned vs detection boxes
runner.run_single_step(pp.step_offset_analysis)


In [ ]:
# unload detector model from GPU to free VRAM before DIFT / PSR optimization
# runner.run_single_step(pp.step_unload_detector) # no need to unload if memory is sufficient

In [ ]:
# Canonical signs setup: resolves period -> eBL Mongo canonical1 images, then EAGERLY DIFT-featurises

runner.run_single_step(pp.step_setup_canonical_signs)

In [ ]:
crop_context._canonical_caches['ebl-mongo:Middle Assyrian:canonical1']


In [ ]:
# create PSR optimizer and plot characteristic loss curves
runner.run_single_step(pp.step_create_psr_optimizer)

In [ ]:
# run PSR optimization, visualize canonical signs at current boxes, probe current boxes, then continue to final
runner.run_single_step(pp.step_run_psr_optimization_until_dift_probe)
runner.run_single_step(pp.step_visualize_canonical_signs_at_psr_boxes)
runner.run_single_step(pp.step_dift_affine_probe)
runner.run_single_step(pp.step_run_psr_optimization_after_dift_probe)

In [ ]:
# optimization loss history
runner.run_single_step(pp.step_plot_loss_history)

In [ ]:
# 2x2 results comparison: coarse aligned, final optimized, det+final overlay, gt+final overlay
runner.run_single_step(pp.step_results_comparison)

In [ ]:
# analyze parameter changes between coarse-aligned and final optimized
runner.run_single_step(pp.step_param_changes)